In [1]:
from mpqp.all import *

In [2]:
from mpqp.execution.result import BatchResult


sampling_devices = [
    GOOGLEDevice.CIRQ_LOCAL_SIMULATOR,
    IBMDevice.AER_SIMULATOR,
    ATOSDevice.MYQLM_CLINALG,
    ATOSDevice.MYQLM_PYLINALG,
    AWSDevice.BRAKET_LOCAL_SIMULATOR,
]

circuit = QCircuit(
    [
        H(0),
        CNOT(0, 1),
        ExpectationMeasure(
            Observable(
                np.array(
                    [
                        [4, 2, 3, 8],
                        [2, -3, 1, 0],
                        [3, 1, -1, 5],
                        [8, 0, 5, 2],
                    ]
                )
            ),
            shots=1024,
        ),
    ]
)
batch = run(circuit, sampling_devices, translation_warning=False)
assert isinstance(batch, BatchResult)
for result in batch:
    print(result.device)
    assert result.error != 0.0
    assert result.shots == 1024

1024
1024
1024
1024
1024


C:\Users\MathieuGras\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\MathieuGras\Desktop\mpqp\mpqp\qasm\qasm_to_braket.py:137: UnsupportedBraketFeaturesWarning: 
This program uses OpenQASM language features that may not be supported on QPUs or on-demand simulators.
  warnings.warn(


GOOGLEDevice.CIRQ_LOCAL_SIMULATOR
IBMDevice.AER_SIMULATOR
ATOSDevice.MYQLM_CLINALG
ATOSDevice.MYQLM_PYLINALG
AWSDevice.BRAKET_LOCAL_SIMULATOR


AssertionError: 

In [2]:
from copy import deepcopy

from mpqp.tools.maths import rand_hermitian_matrix


def circuits_type():
    circuit_state_vector = QCircuit([H(0), CNOT(0, 1)])

    circuit_samples = deepcopy(circuit_state_vector)
    circuit_samples.add(BasisMeasure())

    observable = np.array([[4, 2, 3, 8], [2, -3, 1, 0], [3, 1, -1, 5], [8, 0, 5, 2]])
    circuit_observable = deepcopy(circuit_state_vector)
    circuit_observable.add(ExpectationMeasure(Observable(observable)))

    observable_ideal = rand_hermitian_matrix(2**2)
    circuit_observable_ideal = deepcopy(circuit_state_vector)
    circuit_observable_ideal.add(ExpectationMeasure(Observable(observable_ideal)))
    return [
        circuit_state_vector,
        circuit_samples,
        circuit_observable,
        circuit_observable_ideal,
    ]

In [ ]:
device = GOOGLEDevice.CIRQ_LOCAL_SIMULATOR
print(device.supports_observable())
print(device.supports_samples())
print(device.supports_state_vector())
circuits = circuits_type()

run(circuits[2], device)

In [ ]:
from typing import TYPE_CHECKING
from mpqp.core.instruction.measurement.pauli_string import PauliString
from cirq.circuits.circuit import Circuit as CirqCircuit
from cirq.work.observable_measurement import (
    measure_observables,
    RepetitionsStoppingCriteria,
)
from cirq.sim.sparse_simulator import Simulator
from cirq.ops.pauli_string import PauliString as CirqPauliString

from mpqp.execution.runner import generate_job


def execute_cirq_observable(job: Job, circuit: CirqCircuit):
    variances = {}
    if job.measure is None:
        raise NotImplementedError("job.measure is None")
    assert isinstance(job.measure, ExpectationMeasure)

    if job.measure.optimize_measurement:
        measure: ExpectationMeasure = job.measure
        grouping = job.measure.get_pauli_grouping()

        expectation_values: dict[str, float] = {}
        result: dict[str, float] = {}

        for group in grouping:
            job.status = JobStatus.RUNNING
            if job.measure.shots == 0:
                local_result = Simulator(noise=None).simulate_expectation_values(
                    circuit, observables=cirq_obs
                )
                for i, res in enumerate(local_result):
                    expectation_values.update({f"{group[i].name}": res.real})
                    variances.update({f"{group[i].name}": 0})
            else:
                local_result = measure_observables(
                    circuit,
                    observables=cirq_obs,
                    sampler=Simulator(noise=None),
                    stopping_criteria=RepetitionsStoppingCriteria(job.measure.shots),
                )
                for i, res in enumerate(local_result):
                    expectation_values.update({f"{group[i].name}": res.mean})
                    variances.update({f"{group[i].name}": res.variance})

        errors = {}
        for i, obs in enumerate(measure.observables):
            string = obs.pauli_string
            local: float = 0
            var = {}
            for monoms in string.monomials:
                local += expectation_values[monoms.name]
                var.update({monoms.name: variances[monoms.name]})
            errors.update({f"observable_{i}": var})
            result.update({f"observable_{i}": local})
        if len(result) == 1:
            return Result(job, result["observable_0"], errors["observable_0"])
        return Result(job, result, errors)
    else:
        errors = {}
        expectation_values = {}
        variances = {}
        for obs in job.measure.observables:
            cirq_obs = obs.to_other_language(language=Language.CIRQ, circuit=circuit)
            job.status = JobStatus.RUNNING
            if job.measure.shots == 0:
                results = Simulator(noise=None).simulate_expectation_values(
                    circuit, observables=cirq_obs
                )
                for i, res in enumerate(results):
                    errors.update({f"observable_{len(errors)}": 0})
                    expectation_values.update(
                        {f"observable_{len(expectation_values)}": res.real}
                    )
            else:
                results = measure_observables(
                    circuit,
                    observables=(  # pyright: ignore[reportArgumentType]
                        [cirq_obs]
                        if isinstance(cirq_obs, CirqPauliString)
                        else cirq_obs
                    ),
                    sampler=Simulator(noise=None),
                    stopping_criteria=RepetitionsStoppingCriteria(job.measure.shots),
                )
                pauli_mono = PauliString.from_other_language(
                    [r.observable for r in results], job.measure.nb_qubits
                )
                if TYPE_CHECKING:
                    assert isinstance(pauli_mono, list)
                variances = {pm: r.variance for pm, r in zip(pauli_mono, results)}
                expectation_values.update(
                    {
                        f"observable_{len(expectation_values)}": sum(
                            map(lambda r: r.mean, results)
                        )
                    }
                )
                errors.update({f"observable_{len(errors)}": variances})
    return Result(
        job,
        expectation_values,
        errors,
        job.measure.shots,
    )


circuit = QCircuit(3)
circuit.add(Rx(np.pi / 3, 2))
circuit.add(Ry(np.pi / 12, 1))
circuit.add(H(0))
circuit.add(Rz(np.pi / 4, 0))

from mpqp.measures import *

string = X @ Y @ Z + X @ I @ I + I @ Y @ I + I @ I @ Z
circuit.add(ExpectationMeasure(Observable(string), shots=0))

job = generate_job(circuit, GOOGLEDevice.CIRQ_LOCAL_SIMULATOR)
cirq = circuit.without_measurements().to_other_language(Language.CIRQ)
assert isinstance(cirq, CirqCircuit)
print(execute_cirq_observable(job, cirq))
print(run(circuit, GOOGLEDevice.CIRQ_LOCAL_SIMULATOR))

Result: GOOGLEDevice, CIRQ_LOCAL_SIMULATOR
  Expectation value: 1.207106590270996
  Error/Variance: {'X@I@I': 0, 'I@Y@I': 0, 'I@I@Z': 0, 'X@Y@Z': 0}
JobType.OBSERVABLE
Result: GOOGLEDevice, CIRQ_LOCAL_SIMULATOR
  Expectation value: 1.2071066200733185
  Error/Variance: 0


In [37]:
from braket.circuits import Circuit as BraketCircuit
from braket.tasks import GateModelQuantumTaskResult

from mpqp.execution.connection.aws_connection import get_braket_device


def execute_braket_observable(job: Job, circuit: BraketCircuit):
    device = get_braket_device(job.device, is_noisy=bool(job.circuit.noises))  # type: ignore
    if job.measure is None:
        raise NotImplementedError("job.measure is None")
    assert isinstance(job.measure, ExpectationMeasure)
    results = {}
    errors = {}
    if job.measure.optimize_measurement:
        grouping = job.measure.get_pauli_grouping()
        from mpqp.tools.pauli_grouping import (
            find_qubitwise_rotations,
            pauli_monomial_eigenvalues,
        )

        expectation_values = {}
        for group in grouping:
            pre_measure = QCircuit(find_qubitwise_rotations(group)).to_other_language(
                Language.BRAKET
            )
            job.status = JobStatus.RUNNING
            if job.measure.shots == 0:
                cirq = deepcopy(circuit + pre_measure)
                cirq.state_vector()  # type: ignore
                local_result = device.run(cirq, shots=0, inputs=None)
                sorted_values = local_result.result().values[0]
                for i in range(len(sorted_values)):
                    sorted_values[i] = float(np.abs(sorted_values[i]) ** 2)
            else:
                local_result = device.run(
                    circuit + pre_measure,
                    shots=job.measure.shots,
                    inputs=None,
                )
                result = local_result.result()
                assert isinstance(result, GateModelQuantumTaskResult)
                length = 2**job.circuit.nb_qubits
                sorted_values = []
                for i in range(length):
                    key = f"{bin(i)[2:].zfill(len(bin(length))- 3)}"
                    if key in result.measurement_probabilities:
                        sorted_values.append(result.measurement_probabilities[key])
                    else:
                        sorted_values.append(0)

            for monom in group:
                expectation_value: float = np.dot(
                    pauli_monomial_eigenvalues(monom),
                    np.array(sorted_values, dtype=np.float64),
                )
                expectation_values.update({monom.name: expectation_value})
        for i, obs in enumerate(job.measure.observables):
            string = obs.pauli_string
            local: float = 0
            for monoms in string.monomials:
                assert isinstance(monoms.coef, (int, float))
                local += expectation_values[monoms.name] * monoms.coef
            results.update({f"observable_{i}": local})
            errors.update({f"observable_{len(errors)}": None})
        if len(results) == 1:
            return Result(job, results["observable_0"])
        return Result(job, results, errors)

    else:

        for obs in job.measure.observables:
            copy = deepcopy(circuit)
            braket_obs = obs.to_other_language(Language.BRAKET)
            copy.expectation(  # pyright: ignore[reportAttributeAccessIssue]
                observable=braket_obs, target=job.measure.targets
            )
            job.status = JobStatus.RUNNING
            local_result = device.run(copy, shots=job.measure.shots, inputs=None)
            results.update(
                {f"observable_{len(results)}": local_result.result().values[0]}
            )
            errors.update({f"observable_{len(errors)}": None})
        if len(results) == 1:
            return Result(job, results["observable_0"], None, job.measure.shots)
    return Result(job, results, errors, job.measure.shots)

In [ ]:
circuit = QCircuit(3)
circuit.add(Rx(np.pi / 3, 2))
circuit.add(Ry(np.pi / 12, 1))
circuit.add(H(0))
circuit.add(Rz(np.pi / 4, 0))
circuit.add(Rx(np.pi / 3, 0))
circuit.add(Ry(np.pi / 12, 2))
circuit.add(H(2))
circuit.add(Rz(np.pi / 4, 1))


from mpqp.measures import *


string = X @ Y @ Z + X @ I @ I + I @ Y @ I + I @ I @ Z
str2 = X @ X @ X + Y @ Y @ Y - Z @ Z @ Z


circuit.add(ExpectationMeasure([Observable(string), Observable(str2)], shots=0))



job = generate_job(circuit, AWSDevice.BRAKET_LOCAL_SIMULATOR)
cirq = circuit.without_measurements().to_other_language(Language.BRAKET)
print(execute_braket_observable(job, cirq))

Result: AWSDevice, BRAKET_LOCAL_SIMULATOR
  observable_0:
    Expectation value: 0.6367567773186426
    Error/Variance: None
  observable_1:
    Expectation value: -0.07008252147247757
    Error/Variance: None


C:\Users\MathieuGras\AppData\Local\Temp\ipykernel_13176\2965655326.py:54: ComplexWarning: Casting complex values to real discards the imaginary part
  np.array(sorted_values, dtype=np.float64),


In [43]:
circuit = QCircuit(3)
circuit.add(Rx(np.pi / 3, 2))
circuit.add(Ry(np.pi / 12, 1))
circuit.add(H(0))
circuit.add(Rz(np.pi / 4, 0))
circuit.add(Rx(np.pi / 3, 0))
circuit.add(Ry(np.pi / 12, 2))
circuit.add(H(2))
circuit.add(Rz(np.pi / 4, 1))

from mpqp.measures import *


from mpqp.execution.runner import generate_job


string = X @ Y @ Z + X @ I @ I + I @ Y @ I + I @ I @ Z
str2 = X @ X @ X + Y @ Y @ Y - Z @ Z @ Z
circuit.add(

    ExpectationMeasure(
        [Observable(string), Observable(str2)], shots=0, optimize_measurement=False
    )
)



job = generate_job(circuit, AWSDevice.BRAKET_LOCAL_SIMULATOR)
cirq = circuit.without_measurements().to_other_language(Language.BRAKET)
assert isinstance(cirq, BraketCircuit)
print(execute_braket_observable(job, cirq))

Result: AWSDevice, BRAKET_LOCAL_SIMULATOR
  observable_0:
    Expectation value: 0.6367567773186429
    Error/Variance: None
  observable_1:
    Expectation value: -0.07008252147247773
    Error/Variance: None


In [70]:
print(run(circuit, AWSDevice.BRAKET_LOCAL_SIMULATOR))

JobType.OBSERVABLE
2000
Result: AWSDevice, BRAKET_LOCAL_SIMULATOR
  Expectation value: 0.2260000000000001
  Error/Variance: None


In [ ]:
circuit = QCircuit(3)
circuit.add(Rx(np.pi / 3, 2))
circuit.add(Ry(np.pi / 12, 1))
circuit.add(H(0))
circuit.add(Rz(np.pi / 4, 0))

from mpqp.measures import *

string = X @ Y @ Z + X @ I @ I + I @ Y @ I + I @ I @ Z
str2 = X @ X @ X + Y @ Y @ Y - Z @ Z @ Z
circuit.add(BasisMeasure())

print(run(circuit, AWSDevice.BRAKET_LOCAL_SIMULATOR))

In [ ]:
from cirq.circuits.circuit import Circuit as CirqCircuit
from cirq_google.engine.simulated_local_engine import SimulatedLocalEngine
from cirq_google.engine.simulated_local_processor import SimulatedLocalProcessor
from cirq_google.engine.virtual_engine_factory import (
    create_device_from_processor_id,
    load_median_device_calibration,
)
from qsimcirq.qsim_simulator import QSimSimulator

circuit = QCircuit(3)
circuit.add(Rx(np.pi / 3, 2))
circuit.add(Ry(np.pi / 12, 1))
circuit.add(H(0))
circuit.add(Rz(np.pi / 4, 0))

from mpqp.measures import *

string = X @ Y @ Z + X @ I @ I + I @ Y @ I + I @ I @ Z
str2 = X @ X @ X
circuit.add(
    ExpectationMeasure(
        [Observable(string), Observable(str2)],
        shots=100000,
    )
)

job = generate_job(circuit, GOOGLEDevice.PROCESSOR_RAINBOW)

cirq_circuit = circuit.to_other_device(job.device)
job.circuit.gphase = circuit.gphase

calibration = load_median_device_calibration(job.device.value)
device = create_device_from_processor_id(job.device.value)

# noise_props = noise_properties_from_calibration(cal)
# noise_model = NoiseModelFromGoogleNoiseProperties(noise_props)

simulator = QSimSimulator(noise=None)
sim_processor = SimulatedLocalProcessor(
    processor_id=job.device.value,
    sampler=simulator,
    device=device,
    calibrations={calibration.timestamp // 1000: calibration},
)
simulator = SimulatedLocalEngine([sim_processor])
assert isinstance(cirq_circuit, CirqCircuit)
assert isinstance(job.measure, ExpectationMeasure)
cirq_obs = [
    obs.to_other_language(language=Language.CIRQ, circuit=cirq_circuit)
    for obs in job.measure.observables
]
print(
    simulator.get_sampler(job.device.value).sample_expectation_values(
        cirq_circuit,
        observables=cirq_obs,
        num_samples=job.measure.shots,
    )
)

[[1.54816, -0.00658]]


In [49]:
print(GOOGLEDevice.PROCESSOR_RAINBOW.is_processor())
print(GOOGLEDevice.PROCESSOR_RAINBOW.is_remote())

True
False
